In [1]:
from sentence_transformers import SentenceTransformer, util
import numpy as np

# 1. 加载预训练模型（先从轻量模型入手）
# 推荐新手先试：all-MiniLM-L6-v2（轻量、速度快）
model = SentenceTransformer('all-MiniLM-L6-v2')
# 默认下载路径 C:\Users\【你的电脑用户名】\.cache\huggingface\hub
# 编码维度
# all-MiniLM-L6-v2：384 维（轻量模型，速度快、体积小，最推荐新手）
# all-MiniLM-L12-v2：384 维（和上面同维度，特征提取稍细，速度稍慢）
# all-mpnet-base-v2：768 维（中量级模型，语义表征更精准，体积约 400MB）
# paraphrase-multilingual-MiniLM-L12-v2：384 维（多语言模型，支持中英日韩等，新手做跨语言相似度首选）
# 2. 准备测试文本
texts = [
    "今天天气很好，适合出门散步",
    "今日晴空万里，出门遛弯很合适",
    "人工智能技术正在快速发展",
    "机器学习是人工智能的重要分支"
]

# 3. 生成文本向量（返回numpy数组，shape=(文本数, 向量维度)）
embeddings = model.encode(texts)
print(f"向量维度：{embeddings.shape}")  # all-MiniLM-L6-v2输出384维，输出示例：(4, 384)


向量维度：(4, 384)


d:\Users\jiaendu\Anaconda3\envs\cuda_py3.8\lib\site-packages\transformers\models\bert\modeling_bert.py:440: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


In [4]:

# 4. 计算向量相似度（余弦相似度）
# 计算第0句和第1句的相似度（语义相似）
print(embeddings[0])
sim_0_1 = util.cos_sim(embeddings[0], embeddings[1])
# 计算第0句和第2句的相似度（语义无关）
sim_0_2 = util.cos_sim(embeddings[0], embeddings[2])
sim_0_3 = util.cos_sim(embeddings[0], embeddings[3])
sim_2_3 = util.cos_sim(embeddings[2], embeddings[3])

print(f"句子0和句子1的相似度：{sim_0_1.item():.4f}")  # 约0.8+（高相似）
print(f"句子0和句子2的相似度：{sim_0_2.item():.4f}")  # 约0.1-（低相似）
print(f"句子0和句子3的相似度：{sim_0_3.item():.4f}")  # 约0.1-（低相似）
print(f"句子2和句子3的相似度：{sim_2_3.item():.4f}")  # 约0.1-（低相似）

[-3.37948720e-03  1.04912639e-01  7.60044605e-02  5.23116393e-03
 -6.92253653e-03  7.43317530e-02  9.17100385e-02 -6.29214719e-02
  7.03555942e-02 -1.76945701e-02  8.94243047e-02 -9.30984318e-02
  1.39648002e-03 -9.96906310e-02 -9.14322212e-03 -9.12398398e-02
  1.10404175e-02  5.67621440e-02 -7.32168779e-02 -8.05499498e-03
 -1.92012452e-02  2.44745445e-02 -3.85552790e-04  9.56185386e-02
 -3.42345983e-02  3.18425111e-02 -5.87835070e-03 -2.90604215e-02
  6.69841766e-02  4.07292508e-02 -4.34659198e-02  7.63150081e-02
  7.41558906e-04 -2.10931990e-02  1.56593770e-02  4.65996228e-02
 -5.12448363e-02 -4.95239273e-02 -1.19028799e-02  3.79726961e-02
 -6.46235496e-02 -2.23059785e-02  4.30912226e-02 -2.89569795e-02
  1.35237642e-03  4.35626060e-02 -6.52325079e-02 -8.39182467e-04
  6.89662546e-02  1.31655550e-02 -3.82048935e-02 -2.56074429e-03
 -8.49140063e-02 -1.80649925e-02  6.13262057e-02  1.13729700e-01
 -1.24557661e-02 -8.20961967e-02 -1.36941830e-02 -4.22103852e-02
 -5.26432358e-02  6.89348

`util.cos_sim()`计算的是**余弦相似度（Cosine Similarity）**，这也是Sentence-BERT计算文本语义相似度的**核心方法**。

简单来说，余弦相似度的本质是：**通过计算两个向量之间的夹角余弦值，来衡量向量的相似程度**——文本被编码成向量后，语义越相似，向量在空间中的方向就越接近，夹角越小，余弦值就越大。

### 一、余弦相似度的核心逻辑（通俗版）
#### 1. 计算公式
对于两个n维向量$\vec{a}$和$\vec{b}$，余弦相似度的计算公式为：
$$
\cos(\theta) = \frac{\vec{a} \cdot \vec{b}}{||\vec{a}|| \times ||\vec{b}||}
$$
- 分子$\vec{a} \cdot \vec{b}$：两个向量的**点积**（对应维度相乘后求和）；
- 分母$||\vec{a}|| \times ||\vec{b}||$：两个向量的**模长**（各维度平方和开根号）相乘；
- 结果$\cos(\theta)$：就是向量夹角$\theta$的余弦值。
#### 2. 结果范围与解读（重点）
余弦相似度的结果始终在 **[-1, 1]** 之间，在文本语义相似度场景中，基本只会用到**[0, 1]**（因为Sentence-BERT编码的向量几乎都是正空间的），解读规则：
- 结果**越接近1**：向量夹角越小，文本语义**越相似**（比如你代码里0和1句，结果约0.8+）；
- 结果**越接近0**：向量夹角接近90°，文本语义**几乎无关**（比如你代码里0和2句，结果约0.1-）；
- 结果=0：两个向量完全垂直，无任何语义关联；
- 结果为负：向量方向相反，文本语义完全对立（文本场景极少出现）。

### 二、代码里的计算过程（对应你的示例）
结合你之前的代码，一步步拆解`util.cos_sim(embeddings[0], embeddings[1])`的计算逻辑，让你更直观理解：
#### 步骤1：获取两个向量
你的代码里`embeddings`是(4, 384)的数组，取前两句的向量：
- $\vec{a}$ = embeddings[0] → 384维的句子0向量
- $\vec{b}$ = embeddings[1] → 384维的句子1向量

#### 步骤2：执行余弦相似度计算
`util.cos_sim()`会自动完成3件事，无需你手动计算：
1. 计算$\vec{a}$和$\vec{b}$的**点积**：384个维度各自相乘后求和；
2. 计算$\vec{a}$和$\vec{b}$的**模长**：分别对两个向量的384个维度平方求和后开根号；
3. 用**点积 ÷ 模长乘积**，得到最终的余弦相似度值。

#### 步骤3：结果处理
`util.cos_sim()`返回的是**torch张量**（比如`tensor([[0.8523]])`），所以需要用`.item()`提取其中的数值，方便打印和后续计算。

### 三、补充2个实用知识点
#### 1. 批量计算相似度（比两两计算更高效）
如果有多个文本（比如你的4句），可以直接将整个向量矩阵传入`util.cos_sim()`，会返回**相似度矩阵**（n×n，n为文本数），一次性看到所有文本间的相似度，比循环两两计算快很多：

#### 2. `util.cos_sim()`的优势（为什么不用自己写）
你可以手动用numpy实现余弦相似度，但`sentence_transformers`的`util.cos_sim()`更适合实际开发，原因：
1. **支持批量计算**：直接处理矩阵，无需循环，效率更高；
2. **兼容torch张量/ numpy数组**：输入两种格式都能计算，无需手动转换；
3. **数值稳定**：处理了模长为0的边界情况（避免除以0报错）；
4. **轻量高效**：和Sentence-BERT模型无缝衔接，无需额外引入库。

In [5]:
# 批量计算所有文本间的相似度，返回4×4的相似度矩阵
sim_matrix = util.cos_sim(embeddings, embeddings)
print("所有文本的相似度矩阵：\n", sim_matrix.numpy().round(4))

所有文本的相似度矩阵：
 [[1.     0.7687 0.595  0.4631]
 [0.7687 1.     0.5779 0.4936]
 [0.595  0.5779 1.     0.7898]
 [0.4631 0.4936 0.7898 1.    ]]


### 四、手动实现简易版（理解原理用，实际开发不用）
如果想自己写代码实现余弦相似度，用numpy就能完成，对应单个向量的计算，和`util.cos_sim()`结果一致

In [8]:
import numpy as np

# 手动计算余弦相似度
def cosine_similarity(vec1, vec2):
    # 计算点积
    dot_product = np.dot(vec1, vec2)
    # 计算模长
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)
    # 计算余弦相似度
    return dot_product / (norm1 * norm2)

# 测试
print(cosine_similarity(embeddings[0], embeddings[1]))  # 输出1.0（几乎完全相似）
print(cosine_similarity(embeddings[0], embeddings[2]))  # 输出0.9839（看似数值接近，实际是2维示例，文本场景会更离散）

print(f"句子0和句子1的相似度：{sim_0_1.item():.4f}")  # 约0.8+（高相似）
print(f"句子0和句子2的相似度：{sim_0_2.item():.4f}")  # 约0.1-（低相似）

0.76866657
0.595026
句子0和句子1的相似度：0.7687
句子0和句子2的相似度：0.5950


### 总结
1. 代码中的相似度是**余弦相似度**，由`util.cos_sim()`自动计算，核心是衡量两个向量的夹角；
2. 结果范围**[0,1]**（文本场景），越接近1语义越相似，越接近0越无关；
3. `util.cos_sim()`支持**批量计算**，返回相似度矩阵，是实际开发的最优选择；
4. 余弦相似度的底层是**点积 ÷ 模长乘积**，可以手动实现，但官方工具更高效稳定。